In [ ]:
import importlib
import numpy as np
import matplotlib.pyplot as plt

from adaptive_latents import datasets, proSVD, Pipeline, CenteringTransformer, StreamingKalmanFilter, Bubblewrap, sjPCA, mmICA, ArrayWithTime, plotting_functions, KernelSmoother, VJF
from tqdm.auto import tqdm
from IPython import display
from adaptive_latents.plotting_functions import plot_flow_fields, AnimationManager, plot_history_with_tail
import adaptive_latents
import importlib

from adaptive_latents.predictor import Predictor
from adaptive_latents.regressions import BaseKernelRegressor

In [ ]:
d = datasets.Zong22Dataset()

In [ ]:
d = datasets.Odoherty21Dataset()

In [ ]:
# %matplotlib qt
prosvd_k = 6

p = Pipeline([CenteringTransformer(), KernelSmoother(tau=2*.68/d.neural_data.dt), proSVD(k=prosvd_k)])

labels = ['prosvd','sjpca']
dim_red_methods = [Pipeline(), sjPCA()]
# predictors = [StreamingKalmanFilter(log_level=2, check_dt=True, n_steps_to_predict=1, steps_between_refits=100) for _ in dim_red_methods]
predictors = [Bubblewrap(log_level=2, check_dt=True, n_steps_to_predict=1) for _ in dim_red_methods]
regs = [BaseKernelRegressor(maxlen=10000, length_scale=0.1725) for _ in dim_red_methods]

outputs = [[] for _ in dim_red_methods]

pbar = tqdm(total=round(d.neural_data.t.max(),2))
for data in p.streaming_run_on(d.neural_data):
    # if data.t > 100:
    #     break

    metrics = []
    in_space_data = []
    for dim_red_method, predictor, output_accumulator in zip(dim_red_methods, predictors, outputs):
        in_space_datum = dim_red_method.partial_fit_transform(data)
        in_space_data.append(in_space_datum)
        output_accumulator.append(in_space_datum)

        mse = ((in_space_datum - predictor.predict(1)) ** 2).mean()
        neg_log_pred_p = -predictor.unevaluated_log_pred_p(1)(in_space_datum)
        metrics.append(neg_log_pred_p)
        predictor.partial_fit_transform(in_space_datum)

    best_regressor = np.argmin(metrics)
    for i, (reg, in_space_datum) in enumerate(zip(regs, in_space_data)):
        reg.observe(in_space_datum, np.array([i == best_regressor]))

    pbar.update(round(data.t,2) - pbar.n)


outputs = [ArrayWithTime.from_list(o, drop_early_nans=True, squeeze_type='to_2d') for o in outputs]

In [ ]:
%matplotlib inline
fig, axs = plt.subplots(3, len(dim_red_methods), figsize=(10, 6), squeeze=False, sharex='col', sharey='col', layout='constrained')
x_direction = 2
y_direction = 3

# assert normalize_method in {None, 'none', 'diffs', 'hcubes', 'squares'}
adaptive_latents.plotting_functions.plot_flow_fields(
    {k:v for k,v in zip(labels, outputs)},
    # method='streamplot',
    method='quiver',
    grid_n=20,
    fig=fig, normalize_method='squares', axs=axs[0], format_axis=False,
    x_direction=x_direction, y_direction=y_direction, scatter_alpha=0,
)

for reg, ax in zip(regs, axs[1]):
    ax.scatter(reg.history[:,x_direction], reg.history[:,y_direction], c=reg.history[:,-1], s=1, cmap='plasma', vmin=-.1, vmax=1.1)

for reg, ax, old_ax in zip(regs[:2], axs[2][:2], axs[0][:2]):
    reg:BaseKernelRegressor
    reg.length_scale = .00008
    importlib.reload(adaptive_latents.predictor)
    adaptive_latents.predictor.Predictor.plot_pdf(fig, ax, reg.predict, e1=x_direction, e2=y_direction, xlim=old_ax.get_xlim(), ylim=old_ax.get_ylim(), density=100, native_d=prosvd_k, add_colorbar=False)
    # ax.scatter(reg.history[:,x_direction], reg.history[:,y_direction], c=reg.history[:,-1], s=1, cmap='plasma')
    display.clear_output()
    display.display(fig)


display.clear_output()

In [ ]:
fig, ax = plt.subplots()
for predictor in predictors:
    log_pred_p = ArrayWithTime.from_list(predictor.log['log_pred_p'], drop_early_nans=True, squeeze_type='to_2d')
    ax.plot(log_pred_p.t, log_pred_p)

In [ ]:
o = X
density = 10

fig, ax = plt.subplots()

e1 = np.zeros(2)
e2 = np.zeros(2)
e1[0] = 1
e2[1] = 1

ax.plot(o @ e1, o @ e2, '.')
ax.axis('scaled')
axis = ax.axis()
ax.cla()

x_edges = np.linspace(axis[0], axis[1], density + 1)
y_edges = np.linspace(axis[2], axis[3], density + 1)
x_centers = np.convolve(x_edges, [0.5, 0.5], mode='valid')
y_centers = np.convolve(y_edges, [0.5, 0.5], mode='valid')

# x_grid, y_grid = np.meshgrid(x_edges, y_edges)
# ax.scatter(x_grid.flatten(), y_grid.flatten())
x_grid, y_grid = np.meshgrid(x_centers, y_centers)
# ax.scatter(x_grid.flatten(), y_grid.flatten())

vectors = np.zeros((len(y_centers), len(x_centers), 2))
d_o = np.diff(o, axis=0)
for i in range(len(y_centers)):
    for j in range(len(x_centers)):
        projection = o @ e1
        slice_1 = (x_edges[j] < projection) & (projection < x_edges[j+1])
        projection = o @ e2
        slice_2 = (y_edges[i] < projection) & (projection < y_edges[i+1])
        s = (slice_1 & slice_2)[:-1]
        vectors[i,j,0] = (d_o[s] @ e1).mean()
        vectors[i,j,1] = (d_o[s] @ e2).mean()

ax.streamplot(x_centers, y_centers, vectors[...,0], vectors[...,1])


ax.axis('scaled')
ax.set_xbound(axis[0], axis[1])
ax.set_ybound(axis[2], axis[3])



In [ ]:
# for reg, ax in zip(regs, axs[2]):
#     reg:BaseKernelRegressor
#     length_scales = np.logspace(-3.5, .5, 20)
#     best_scale, (length_scales, errors, error_stds) = reg.cross_validate_length_scale(length_scales, depth=500, n_train=1, ratio=None)
#     reg.length_scale = best_scale
#     # reg.length_scale = 0.17252105499420392
#     ax.plot(length_scales, errors+error_stds)
#     ax.axvline(best_scale, color='k')
#     ax.semilogx()
#
#     display.clear_output()
#     display.display(fig)
#
